# Predicción del Nutri-Score con Open Food Facts — CRISP-ML(Q)

**Objetivo:** clasificar el grado Nutri-Score (A–E) de un producto a partir de sus nutrientes.
**Dataset:** Open Food Facts (Kaggle `openfoodfacts/world-food-facts`), ~356 mil productos y 163 columnas.

| Fase CRISP-ML(Q) | Sección del notebook |
|---|---|
| 1. Comprensión del negocio y de los datos | §1 y §2 |
| 2. Preparación de datos | §3 |
| 3. Modelado | §4 |
| 4. Evaluación (con aseguramiento de calidad) | §5 y §6 (experimento sin fórmula) |
| 5. Despliegue | §7 (artefacto versionado que consume la plataforma) |
| 6. Monitoreo y mantenimiento | Fuera del notebook: `retrain_service` (ver `docs/ml-lifecycle.md`) |

**Todo el código de datos y modelos vive en el paquete `nutriscore` (`ml/nutriscore`)**: este notebook solo lo
orquesta y lo explica. Así, la limpieza, el preprocesamiento y la evaluación son exactamente los mismos que
usan el servicio de reentrenamiento y la API.

**Cómo ejecutarlo**
- *Colab:* Entorno de ejecución → Ejecutar todas. La primera celda clona el repositorio e instala el paquete.
  Si avisa que hay versiones viejas cargadas, reinicia la sesión (Entorno de ejecución → Reiniciar sesión)
  y vuelve a ejecutar todo.
- *Local:* `pip install -e "ml[notebook]"` desde la raíz del repositorio y abre este archivo.
- Variables de entorno opcionales: `NUTRISCORE_RAW` (ruta al TSV), `NUTRISCORE_PROCESSED` (parquet de salida),
  `NUTRISCORE_MODELS_DIR` (carpeta de modelos).

## 0. Entorno reproducible

In [ ]:
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AlvaroThg/final_openfood.git"

if IN_COLAB:
    # La carpeta del clon se llama como el repositorio (final_openfood). Ruta absoluta: la celda
    # se puede volver a ejecutar aunque el directorio actual ya haya cambiado.
    REPO_DIR = Path("/content") / REPO_URL.rstrip("/").removesuffix(".git").rsplit("/", 1)[-1]
    if REPO_DIR.exists() and not (REPO_DIR / "ml").exists():
        shutil.rmtree(REPO_DIR)  # clon vacío o incompleto de un intento anterior
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "ml[notebook]"], check=True)
    # Un `pip install -e` no se ve en el proceso que ya está corriendo: se agrega la ruta a mano.
    sys.path.insert(0, str(REPO_DIR / "ml"))
    importlib.invalidate_caches()
    # Si Colab ya tenía cargada otra versión de una librería, la seguiría usando hasta reiniciar.
    stale = [
        dist
        for dist, module in [("numpy", "numpy"), ("pandas", "pandas"), ("scikit-learn", "sklearn"),
                             ("lightgbm", "lightgbm"), ("joblib", "joblib")]
        if module in sys.modules and sys.modules[module].__version__ != importlib.metadata.version(dist)
    ]
    if stale:
        raise RuntimeError(
            f"Colab tenía cargadas otras versiones de {stale}. Ve a Entorno de ejecución → "
            "Reiniciar sesión y vuelve a ejecutar desde esta celda."
        )

ROOT = Path.cwd() if (Path.cwd() / "ml").exists() else Path.cwd().resolve().parents[1]
assert (ROOT / "ml" / "nutriscore").exists(), f"No se encontró el repositorio desde {Path.cwd()}"
print("Raíz del repositorio:", ROOT)


In [ ]:
# Fail fast: las versiones instaladas deben ser EXACTAMENTE las fijadas en ml/pyproject.toml.
# Un modelo entrenado con otras versiones no podrá cargarse en la plataforma.
import tomllib

from nutriscore.artifacts import dependency_versions

pins = dict(
    dep.split("==") for dep in tomllib.loads((ROOT / "ml" / "pyproject.toml").read_text())["project"]["dependencies"]
)
installed = dependency_versions()
mismatch = {pkg: (pin, installed.get(pkg)) for pkg, pin in pins.items() if installed.get(pkg) != pin}
assert not mismatch, f"Versiones distintas a las fijadas (fijada, instalada): {mismatch}. Reinicia la sesión."
installed

In [ ]:
# Estilo de gráficos: superficie clara, tinta neutra, un solo tono para magnitudes.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

INK, INK_2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
BLUE = "#2a78d6"
BLUES = LinearSegmentedColormap.from_list(
    "blues", ["#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#104281", "#0d366b"]
)
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": ["Segoe UI", "DejaVu Sans", "sans-serif"], "font.size": 10,
    "text.color": INK, "axes.labelcolor": INK_2, "axes.titlecolor": INK, "axes.titlesize": 11,
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.edgecolor": AXIS,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
})
pd.set_option("display.float_format", "{:.4f}".format)


def hbar(values: pd.Series, title: str, xlabel: str, xlim: tuple[float, float] | None = None) -> None:
    # Barras horizontales de un solo tono, ordenadas, con separación de 2 px.
    values = values.sort_values()
    fig, ax = plt.subplots(figsize=(7, 0.28 * len(values) + 1))
    ax.barh(values.index, values.to_numpy(), color=BLUE, height=0.7, edgecolor=SURFACE, linewidth=2)
    ax.set_title(title, loc="left")
    ax.set_xlabel(xlabel)
    ax.grid(axis="y", visible=False)
    if xlim:
        ax.set_xlim(*xlim)
    plt.tight_layout()
    plt.show()


def plot_confusions(reports: dict[str, dict], title: str) -> None:
    # Matrices de confusión normalizadas por fila (= recall por clase), misma escala para comparar.
    fig, axes = plt.subplots(1, len(reports), figsize=(4.2 * len(reports), 4), squeeze=False)
    for ax, (name, report) in zip(axes[0], reports.items()):
        labels = report["confusion_matrix"]["labels"]
        counts = np.array(report["confusion_matrix"]["matrix"], dtype=float)
        share = counts / counts.sum(axis=1, keepdims=True)
        image = ax.imshow(share, cmap=BLUES, vmin=0, vmax=1)
        for i in range(len(labels)):
            for j in range(len(labels)):
                ax.text(j, i, f"{share[i, j]:.0%}", ha="center", va="center", fontsize=8,
                        color="#ffffff" if share[i, j] > 0.55 else INK)
        ax.set_xticks(range(len(labels)), labels)
        ax.set_yticks(range(len(labels)), labels)
        ax.set_xlabel("Predicho")
        ax.set_ylabel("Real")
        ax.set_title(f"{name}\nF1 macro {report['f1_macro']:.3f}", loc="left")
        ax.grid(False)
    fig.colorbar(image, ax=axes[0].tolist(), shrink=0.8, label="Proporción de la clase real")
    fig.suptitle(title, x=0.01, ha="left", color=INK)
    plt.show()

## 1. Comprensión del negocio

- **Problema:** estimar el Nutri-Score de un producto cuando solo se conocen sus nutrientes por 100 g.
- **Tipo de tarea:** clasificación multiclase ordinal (A–E), con clases desbalanceadas.
- **Criterios de éxito (política de calidad `qp-1`, `ml/nutriscore/quality.py`):** F1 macro, precisión y
  recall **por clase** (ninguna clase abandonada), tamaño del artefacto ≤ 20 MB (hosting modesto) y paridad
  de predicciones tras guardar y recargar. Nunca se aprueba por una sola métrica global.
- **Riesgos:** volumen (≈1 GB), muchos nulos, fuga de datos (el puntaje numérico determina el grado),
  diferencias de preprocesamiento entre entrenamiento y API, tamaño del modelo.
- **Advertencia científica:** el Nutri-Score *es una fórmula* sobre energía, azúcares, grasas saturadas, sal,
  fibra, proteínas y % de frutas/verduras. Un modelo con esas variables puede tener exactitud alta porque
  **aproxima la fórmula**, no porque descubra algo nuevo. El experimento 2 (§6) mide cuánto se predice sin ellas.

## 2. Comprensión de los datos

In [ ]:
from nutriscore import schema
from nutriscore.cleaning import iter_raw_chunks

RAW = Path(os.environ.get("NUTRISCORE_RAW", ROOT / "data" / "raw" / "en.openfoodfacts.org.products.tsv"))
if not RAW.exists():
    import kagglehub  # descarga pública; si pide credenciales, configura KAGGLE_USERNAME y KAGGLE_KEY

    RAW = Path(kagglehub.dataset_download("openfoodfacts/world-food-facts")) / "en.openfoodfacts.org.products.tsv"

header = pd.read_csv(RAW, sep="\t", nrows=0).columns
print(f"Archivo: {RAW.name} · {RAW.stat().st_size / 1024**2:,.0f} MB · {len(header)} columnas")
print("Columna objetivo presente:", schema.RAW_TARGET in header)
print("Columnas de fuga presentes (se excluyen):", [c for c in schema.LEAKAGE_RAW_COLUMNS if c in header])
print("Columnas del contrato que faltan:", sorted(set(schema.RAW_COLUMNS) - set(header)) or "ninguna")

In [ ]:
# Lectura por bloques: solo las ~22 columnas del contrato, en trozos de 50 mil filas.
rows_total = rows_with_grade = 0
present = pd.Series(0, index=list(schema.RAW_COLUMNS))
grades = pd.Series(dtype="int64")
for chunk in iter_raw_chunks(RAW, chunksize=50_000):
    rows_total += len(chunk)
    with_grade = chunk[chunk[schema.RAW_TARGET].notna()]
    rows_with_grade += len(with_grade)
    present = present.add(with_grade.notna().sum(), fill_value=0)
    grades = grades.add(with_grade[schema.RAW_TARGET].str.upper().value_counts(), fill_value=0)

print(f"Filas totales: {rows_total:,} · con grado: {rows_with_grade:,} ({rows_with_grade / rows_total:.0%})")
coverage = (present / rows_with_grade * 100).drop([schema.ID_COLUMN, schema.RAW_TARGET])
coverage.index = [schema.RAW_TO_CANONICAL[c] for c in coverage.index]
hbar(coverage, "Cobertura de cada feature (productos con grado)", "% de filas con dato", (0, 100))

In [ ]:
grade_counts = grades.reindex(list(schema.GRADES)).fillna(0).astype(int)
fig, ax = plt.subplots(figsize=(6, 3))
positions = np.arange(len(grade_counts))
ax.bar(positions, grade_counts.to_numpy(), color=BLUE, width=0.6, edgecolor=SURFACE, linewidth=2)
ax.set_xticks(positions, grade_counts.index)
for x, value in zip(positions, grade_counts.to_numpy()):
    ax.text(x, value, f"{value:,}", ha="center", va="bottom", fontsize=8, color=INK_2)
ax.set_title("Distribución del grado Nutri-Score", loc="left")
ax.set_ylabel("Productos")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()
print("Proporciones:", (grade_counts / grade_counts.sum()).round(3).to_dict())

### 2.1 Demostración de la fuga de datos

Si el grado se obtiene del puntaje numérico por umbrales, cada grado ocupa un rango de puntaje casi sin
solaparse. Por eso `nutrition-score-fr_100g` y `nutrition-score-uk_100g` **se excluyen** del modelo.

In [ ]:
score = pd.concat(
    chunk.dropna()
    for chunk in pd.read_csv(RAW, sep="\t", usecols=[schema.RAW_TARGET, "nutrition-score-fr_100g"],
                             dtype=str, chunksize=50_000)
)
score["grade"] = score[schema.RAW_TARGET].str.upper()
score["points"] = pd.to_numeric(score["nutrition-score-fr_100g"], errors="coerce")
score.groupby("grade")["points"].describe()[["count", "min", "25%", "50%", "75%", "max"]]

**Lectura:** los rangos de puntaje por grado son prácticamente disjuntos (los solapes vienen de las reglas
especiales para bebidas, quesos y grasas añadidas). Un modelo con el puntaje "acertaría" sin aprender nada.

## 3. Preparación de los datos

In [ ]:
from nutriscore.cleaning import load_clean_dataset
from nutriscore.splits import split_frame, stratified_sample

PROCESSED = Path(os.environ.get("NUTRISCORE_PROCESSED", ROOT / "data" / "processed" / "clean_v1.parquet"))
clean, report = load_clean_dataset(RAW, chunksize=50_000)
pd.Series(report.to_dict(), dtype=object).to_frame("valor")

In [ ]:
# Dataset inmutable: se escribe una sola vez. Es el archivo que se sube a retrain_service como "v1".
if PROCESSED.exists():
    print(f"{PROCESSED.name} ya existe; no se sobrescribe (versión inmutable).")
else:
    PROCESSED.parent.mkdir(parents=True, exist_ok=True)
    clean.to_parquet(PROCESSED, index=False)
    print(f"Guardado {PROCESSED} · {PROCESSED.stat().st_size / 1024**2:.1f} MB")

In [ ]:
# Partición determinista por hash del código: 70 % train · 15 % val · 15 % test.
# Un producto cae siempre en la misma partición, también en versiones futuras del dataset.
splits = split_frame(clean)
pd.DataFrame({
    name: part[schema.TARGET].value_counts(normalize=True).reindex(list(schema.GRADES))
    for name, part in splits.items()
}).T.assign(filas=[len(p) for p in splits.values()])

In [ ]:
# Muestra estratificada para explorar y comparar modelos rápido (50-100 mil filas).
SAMPLE_SIZE, SEED = 80_000, 42
sample = stratified_sample(splits["train"], SAMPLE_SIZE, seed=SEED)
exploration = {"train": sample, "val": splits["val"], "test": splits["test"]}
print(f"Muestra de entrenamiento: {len(sample):,} filas · validación: {len(splits['val']):,} filas")

## 4. Modelado

Tres modelos sobre **el mismo pipeline** (imputación por mediana + indicador de faltante, normalización de
categorías + one-hot), todos con **pesos de clase** por el desbalance:

| Modelo | Rol | Configuración |
|---|---|---|
| Árbol de decisión | línea base interpretable | `ml/configs/decision_tree.toml` |
| Random Forest | ensamble por bagging | `ml/configs/random_forest.toml` |
| LightGBM | gradient boosting (candidato a producción) | `ml/configs/lightgbm.toml` |

Se entrena con la muestra y se compara en **validación**. El test no se mira hasta el final (§7).

In [ ]:
import io

import joblib

from nutriscore.quality import CURRENT_POLICY, evaluate_gate
from nutriscore.training import TrainingConfig, train_and_evaluate


def artifact_bytes(pipeline) -> int:
    buffer = io.BytesIO()
    joblib.dump(pipeline, buffer, compress=3)  # misma compresión que el artefacto real
    return buffer.getbuffer().nbytes


CONFIGS = {name: TrainingConfig.from_toml(ROOT / "ml" / "configs" / f"{name}.toml")
           for name in ("decision_tree", "random_forest", "lightgbm")}
results, sizes, gates = {}, {}, {}
for name, config in CONFIGS.items():
    results[name] = train_and_evaluate(config, exploration, evaluate_test=False,
                                       min_slice_support=CURRENT_POLICY.min_slice_support)
    sizes[name] = artifact_bytes(results[name].pipeline)
    # La paridad se verifica al guardar el artefacto final (§7); aquí se asume para comparar.
    gates[name] = evaluate_gate(results[name].validation, active=None,
                                artifact_bytes=sizes[name], parity_ok=True)
    print(f"{name}: entrenado en {results[name].duration_s:.1f} s")

In [ ]:
comparison = pd.DataFrame({
    name: {
        "accuracy": r.validation["accuracy"],
        "f1_macro": r.validation["f1_macro"],
        "precision_macro": r.validation["precision_macro"],
        "recall_macro": r.validation["recall_macro"],
        "recall_min_clase": min(c["recall"] for c in r.validation["per_class"].values()),
        "tamaño_MB": sizes[name] / 1024**2,
        "entrenamiento_s": r.duration_s,
        "aprobaría_qp1": gates[name].passed,
        "criterios_fallidos": ", ".join(c.name for c in gates[name].failed_blocking) or "—",
    }
    for name, r in results.items()
}).T
comparison

In [ ]:
# Recall por clase: ¿alguna clase queda abandonada?
pd.DataFrame({name: {g: m["recall"] for g, m in r.validation["per_class"].items()}
              for name, r in results.items()}).T

## 5. Evaluación

In [ ]:
plot_confusions({name: r.validation for name, r in results.items()}, "Validación · recall por clase en la diagonal")

In [ ]:
# Rendimiento por grupo de alimento (evaluación de sesgo): ¿el modelo falla en algún tipo de producto?
best_name = "lightgbm"  # <- ajustar según la tabla de comparación
groups = results[best_name].validation["slices"]["groups"]
if groups:
    display(pd.DataFrame(groups).T.sort_values("f1_macro"))
else:
    print("Ningún grupo alcanza el soporte mínimo de", CURRENT_POLICY.min_slice_support, "filas")

In [ ]:
# Importancia de variables (ganancia) del modelo LightGBM.
lgbm_pipeline = results["lightgbm"].pipeline
importance = pd.Series(
    lgbm_pipeline.named_steps["model"].booster_.feature_importance(importance_type="gain"),
    index=lgbm_pipeline.named_steps["preprocess"].get_feature_names_out(),
)
importance = importance / importance.sum() * 100
hbar(importance.nlargest(15), "LightGBM · importancia de variables (top 15)", "% de la ganancia total")
formula_share = importance[[c for c in importance.index if c in schema.FORMULA_FEATURES]].sum()
print(f"Las variables de la fórmula concentran el {formula_share:.0f} % de la importancia.")

## 6. Experimento 2: sin las variables de la fórmula

Se repite LightGBM **sin** energía, grasas saturadas, azúcares, sal, fibra, proteínas ni frutas/verduras
(`ml/configs/lightgbm_sin_formula.toml`). Responde: *¿cuánto del grado se puede predecir sin copiar la fórmula?*
Ojo: grasas totales y carbohidratos siguen correlacionados con grasas saturadas y azúcares (variables proxy),
así que no es una eliminación "pura" de la información de la fórmula.

In [ ]:
exp2_config = TrainingConfig.from_toml(ROOT / "ml" / "configs" / "lightgbm_sin_formula.toml")
exp2 = train_and_evaluate(exp2_config, exploration, evaluate_test=False,
                          min_slice_support=CURRENT_POLICY.min_slice_support)
pd.DataFrame({
    "con fórmula": {k: results["lightgbm"].validation[k] for k in ("accuracy", "f1_macro", "recall_macro")},
    "sin fórmula": {k: exp2.validation[k] for k in ("accuracy", "f1_macro", "recall_macro")},
}).T

In [ ]:
plot_confusions({"Con fórmula": results["lightgbm"].validation, "Sin fórmula": exp2.validation},
                "Experimento 2 · validación")

## 7. Modelo final y artefacto

El modelo elegido se reentrena con **todo el split de entrenamiento del conjunto limpio** (no con la muestra),
se evalúa en validación (decide la política de calidad) y en test **una sola vez** (solo se reporta).

> Interpretación de "entrenar con todo el conjunto limpio": validación y test se mantienen fuera del
> entrenamiento para poder medir honestamente y aplicar la política de calidad (AGENTS.md §5).

Se usa el mismo script que reproduce cualquier versión fuera de la plataforma (`ml/scripts/train.py`).
En producción, este mismo flujo lo ejecuta `retrain_worker` con el dataset `v1` subido.

In [ ]:
import json

MODELS = Path(os.environ.get("NUTRISCORE_MODELS_DIR", ROOT / "models"))
CHOSEN = "lightgbm"  # <- configuración elegida en §4-§5
VERSION = "1.0.0"
if (MODELS / f"v{VERSION}").exists():
    print(f"La versión {VERSION} ya existe en {MODELS}; las versiones son inmutables.")
else:
    completed = subprocess.run(
        [sys.executable, str(ROOT / "ml" / "scripts" / "train.py"),
         "--config", str(ROOT / "ml" / "configs" / f"{CHOSEN}.toml"),
         "--data", str(PROCESSED), "--dataset-version", "v1",
         "--models-dir", str(MODELS), "--version", VERSION],
        capture_output=True, text=True,
    )
    print(completed.stdout or completed.stderr)
    assert completed.returncode in (0, 3), "El entrenamiento falló (ver salida)"

In [ ]:
final_metrics = json.loads((MODELS / f"v{VERSION}" / "metrics.json").read_text(encoding="utf-8"))
final_gate = json.loads((MODELS / f"v{VERSION}.gate.json").read_text(encoding="utf-8"))
print("Política", final_gate["policy_version"], "· aprobado:", final_gate["passed"])
pd.DataFrame({split: {k: final_metrics[split][k] for k in ("accuracy", "f1_macro", "precision_macro", "recall_macro")}
              for split in ("validation", "test")}).T

In [ ]:
plot_confusions({"Validación": final_metrics["validation"], "Test (una sola vez)": final_metrics["test"]},
                f"Modelo final v{VERSION} · {CHOSEN}")

## 8. Conclusiones *(completar tras ejecutar)*

1. **Modelo elegido y por qué:** _[métricas de validación, tamaño, criterios de qp-1]_.
2. **Exactitud "inflada":** el modelo con las variables de la fórmula alcanza _[F1]_ porque aproxima una regla
   conocida; sus errores se concentran en _[clases vecinas, p. ej. C↔D]_, explicables por datos faltantes
   (fibra, % de frutas), reglas especiales por categoría (bebidas, quesos, grasas) y cambios de la fórmula en el tiempo.
3. **Experimento 2:** sin la fórmula el F1 macro cae a _[valor]_; la información restante (categoría, aditivos,
   aceite de palma, grasas totales y carbohidratos como proxies) explica _[interpretación]_.
4. **Limitaciones:** dataset de 2017-2018, sesgo geográfico (muchos productos de EE. UU. y Francia),
   grupos con F1 bajo en §5 _[listar]_.
5. **Siguiente paso:** el dataset `v1` y la configuración elegida se cargan en la plataforma; `retrain_service`
   reproduce el entrenamiento y la política de calidad decide su aprobación.